# Module 07 — World Models & Environment Modeling

> **SDKs:** `sqlite3`, `pydantic`, `dataclasses`

| Part | Topic |
|------|-------|
| **1** | Digital Twins — sandboxed pre-execution |
| **2** | Counterfactual Planning — Tree of Thoughts |
| **3** | Sim-to-Real Gap — calibrated sensors |


---
## Part 1 — Digital Twins: Sandboxed Pre-Execution

A dangerous agent calls `DROP TABLE production.orders` directly. A safe agent runs it against a `sqlite3` in-memory clone first.

In [1]:
import sqlite3, copy, time
from dataclasses import dataclass, field
from typing import Any, Optional

class DigitalTwin:
    """
    In-memory SQLite replica of the production schema.
    The agent MUST run destructive operations here first.
    Only if the twin succeeds do we propose the action on production.
    """
    def __init__(self, schema_sql: str, seed_sql: str = ""):
        self.conn = sqlite3.connect(":memory:")
        self.conn.row_factory = sqlite3.Row
        self.conn.executescript(schema_sql)
        if seed_sql:
            self.conn.executescript(seed_sql)
        self.conn.commit()

    def execute(self, sql: str) -> list[dict]:
        try:
            cur = self.conn.execute(sql)
            self.conn.commit()
            rows = cur.fetchall()
            return [dict(r) for r in rows]
        except sqlite3.Error as e:
            raise RuntimeError(f"Twin SQL error: {e}")

    def snapshot_count(self, table: str) -> int:
        return self.conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

SCHEMA = """
CREATE TABLE orders (
    id      INTEGER PRIMARY KEY,
    tenant  TEXT NOT NULL,
    amount  REAL NOT NULL,
    status  TEXT NOT NULL
);
"""
SEED = """
INSERT INTO orders VALUES (1,'northstar-eu-001', 4200.00,'pending');
INSERT INTO orders VALUES (2,'northstar-eu-001', 800.00,'completed');
INSERT INTO orders VALUES (3,'globex-002',       5500.00,'pending');
INSERT INTO orders VALUES (4,'acme-003',         1200.00,'failed');
"""

twin = DigitalTwin(SCHEMA, SEED)

print("🔬  Digital Twin Demo")
print("=" * 60)
print(f"  Twin initialized with {twin.snapshot_count('orders')} orders")

# Simulate agent wanting to delete failed orders
sql_proposal = "DELETE FROM orders WHERE status = 'failed'"
print(f"\n  Agent wants to execute: {sql_proposal!r}")
print(f"  Running on TWIN first...")

before = twin.snapshot_count("orders")
twin.execute(sql_proposal)
after = twin.snapshot_count("orders")

print(f"  ✅  Twin result: {before} → {after} rows (deleted {before-after})")
print(f"  Twin verified safe. Now generating proposal for human approval.")
print(f"\n  🛑  Agent STOPS HERE. Does NOT execute on Production.")
print(f"  Proposal: delete {before-after} failed-status orders in production.orders")


🔬  Digital Twin Demo
  Twin initialized with 4 orders

  Agent wants to execute: "DELETE FROM orders WHERE status = 'failed'"
  Running on TWIN first...
  ✅  Twin result: 4 → 3 rows (deleted 1)
  Twin verified safe. Now generating proposal for human approval.

  🛑  Agent STOPS HERE. Does NOT execute on Production.
  Proposal: delete 1 failed-status orders in production.orders


---
## Part 2 — Counterfactual Planning: Tree of Thoughts

Before committing to an action, the agent simulates multiple divergent futures in the Digital Twin and selects the highest utility path.

In [2]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ActionNode:
    action: str
    description: str
    simulated_outcome: dict
    expected_utility: float    # -1.0 to 1.0
    risk_score: float          # 0.0 to 1.0

def simulate_action(action: str, twin: DigitalTwin) -> dict:
    """Run action on Digital Twin and return state delta."""
    if action == "immediate_revert":
        return {"downtime_saved_min": 22, "data_loss": False, "rollback_risk": "LOW",  "mttr_min": 3}
    elif action == "gradual_rollout":
        return {"downtime_saved_min": 10, "data_loss": False, "rollback_risk": "LOW",  "mttr_min": 15}
    elif action == "wait_and_monitor":
        return {"downtime_saved_min": 0,  "data_loss": False, "rollback_risk": "NONE", "mttr_min": 60}
    elif action == "database_rollback":
        return {"downtime_saved_min": 20, "data_loss": True,  "rollback_risk": "HIGH", "mttr_min": 5}
    return {}

def score_utility(outcome: dict) -> tuple[float, float]:
    """Score utility and risk from a simulated outcome."""
    utility = (outcome.get("downtime_saved_min", 0) / 30)
    risk = 0.8 if outcome.get("data_loss") else 0.1
    if outcome.get("rollback_risk") == "HIGH": risk += 0.4
    return min(utility, 1.0), min(risk, 1.0)

actions = ["immediate_revert", "gradual_rollout", "wait_and_monitor", "database_rollback"]
nodes = []
print("🌳  Counterfactual Planning (Tree of Thoughts)")
print("=" * 60)

for action in actions:
    outcome = simulate_action(action, twin)
    utility, risk = score_utility(outcome)
    node = ActionNode(
        action=action, description=action.replace("_", " ").title(),
        simulated_outcome=outcome, expected_utility=utility, risk_score=risk,
    )
    nodes.append(node)
    icon = "💡" if (utility > 0.5 and risk < 0.4) else ("⚠️" if risk > 0.5 else "·")
    print(f"  {icon}  {action:<25} utility={utility:.2f}  risk={risk:.2f}  "
          f"mttr={outcome.get('mttr_min')}min  data_loss={outcome.get('data_loss')}")

best = max(nodes, key=lambda n: n.expected_utility - n.risk_score)
print(f"\n  🏆  Selected action: {best.action}")
print(f"       Utility: {best.expected_utility:.2f}  Risk: {best.risk_score:.2f}")
print(f"       MTTR   : {best.simulated_outcome['mttr_min']} minutes")


🌳  Counterfactual Planning (Tree of Thoughts)
  💡  immediate_revert          utility=0.73  risk=0.10  mttr=3min  data_loss=False
  ·  gradual_rollout           utility=0.33  risk=0.10  mttr=15min  data_loss=False
  ·  wait_and_monitor          utility=0.00  risk=0.10  mttr=60min  data_loss=False
  ⚠️  database_rollback         utility=0.67  risk=1.00  mttr=5min  data_loss=True

  🏆  Selected action: immediate_revert
       Utility: 0.73  Risk: 0.10
       MTTR   : 3 minutes


---
## Part 3 — The Sim-to-Real Gap: Calibrated Sensors

The Digital Twin assumes 10ms latency. Reality is 4200ms. If the agent doesn't calibrate its model with live sensor data before acting, it will make catastrophically wrong predictions.

In [3]:
import random, time
from dataclasses import dataclass

@dataclass
class Sensor:
    name: str
    assumed_ms: float   # What the world model assumes
    actual_ms: float    # What the real system measures

    @property
    def gap_pct(self) -> float:
        return abs(self.actual_ms - self.assumed_ms) / self.assumed_ms * 100

    @property
    def is_calibrated(self) -> bool:
        return self.gap_pct < 20   # Within 20% tolerance

sensors = [
    Sensor("3DS gateway latency",    assumed_ms=10,    actual_ms=4200),
    Sensor("Postgres query p99",     assumed_ms=5,     actual_ms=45),
    Sensor("Redis SET latency",      assumed_ms=1,     actual_ms=2),
    Sensor("Stripe API call",        assumed_ms=200,   actual_ms=350),
    Sensor("Feature flag toggle",    assumed_ms=50,    actual_ms=48),
]

print("📡  Sim-to-Real Gap Analysis")
print("=" * 65)
print(f"  {'Sensor':<30} {'Assumed':<12} {'Actual':<12} {'Gap%':<10} {'Status'}")
print(f"  {'─'*30} {'─'*12} {'─'*12} {'─'*10} {'─'*15}")

critical_gaps = []
for s in sensors:
    status = "✅ OK" if s.is_calibrated else "❌ CRITICAL"
    if not s.is_calibrated:
        critical_gaps.append(s)
    print(f"  {s.name:<30} {s.assumed_ms:<12.0f} {s.actual_ms:<12.0f} {s.gap_pct:<10.0f}% {status}")

print(f"\n  ⚠️  Critical gaps found: {len(critical_gaps)}")
for g in critical_gaps:
    print(f"    [{g.name}] Model assumes {g.assumed_ms}ms — reality is {g.actual_ms}ms ({g.gap_pct:.0f}% error)")
    print(f"    → World model predictions are INVALID for this sensor.")
    print(f"    → Agent must re-calibrate or use conservative timeouts.")


📡  Sim-to-Real Gap Analysis
  Sensor                         Assumed      Actual       Gap%       Status
  ────────────────────────────── ──────────── ──────────── ────────── ───────────────
  3DS gateway latency            10           4200         41900     % ❌ CRITICAL
  Postgres query p99             5            45           800       % ❌ CRITICAL
  Redis SET latency              1            2            100       % ❌ CRITICAL
  Stripe API call                200          350          75        % ❌ CRITICAL
  Feature flag toggle            50           48           4         % ✅ OK

  ⚠️  Critical gaps found: 4
    [3DS gateway latency] Model assumes 10ms — reality is 4200ms (41900% error)
    → World model predictions are INVALID for this sensor.
    → Agent must re-calibrate or use conservative timeouts.
    [Postgres query p99] Model assumes 5ms — reality is 45ms (800% error)
    → World model predictions are INVALID for this sensor.
    → Agent must re-calibrate or use conser